<a href="https://colab.research.google.com/github/ibarr123/BUS1182026/blob/main/Group_Customer_Service_Bot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re
from google import genai
from google.colab import userdata

api_key = userdata.get("OPENAI_API_KEY")
client = genai.Client(api_key=api_key)

print("API key loaded successfully.")

API key loaded successfully.


In [ ]:
system_prompt = """
You are TechNest's customer service and sales assistant.

Your job is to help customers with:
1. Order status questions
2. Refund policy questions
3. Product recommendations

Rules:
- Be friendly, helpful, and concise.
- Maintain context across the conversation.
- If the user asks about order status, ask for or use their order number.
- Refund policy: Returns are accepted within 30 days of delivery if the item is unused and in the original packaging.
- Opened items may qualify for store credit depending on condition.
- For product recommendations, consider the user's budget, intended use, and preferences.
- If the request is outside your role, politely explain that you specialize in store support and product suggestions.
"""

refund_policy_text = """
TechNest Refund Policy:
- Returns accepted within 30 days of delivery
- Item must be unused and in original packaging for full refund
- Opened items may be eligible for store credit depending on condition
- Final sale items cannot be returned
"""

products = [
    {
        "name": "FlexBeat Lite Earbuds",
        "category": "earbuds",
        "price": 79,
        "features": ["wireless", "sweat-resistant", "lightweight"],
        "best_for": "gym"
    },
    {
        "name": "AeroPods Fit",
        "category": "earbuds",
        "price": 99,
        "features": ["wireless", "secure fit", "noise reduction"],
        "best_for": "gym"
    },
    {
        "name": "StudyPro 14 Laptop",
        "category": "laptop",
        "price": 749,
        "features": ["lightweight", "long battery life", "student-friendly"],
        "best_for": "school"
    },
    {
        "name": "PowerBook X",
        "category": "laptop",
        "price": 999,
        "features": ["fast processor", "16GB RAM", "portable"],
        "best_for": "school"
    },
    {
        "name": "WorkFlow Monitor 27",
        "category": "monitor",
        "price": 229,
        "features": ["27-inch", "HD display", "good for productivity"],
        "best_for": "office"
    }
]

order_database = {
    "1001": "Order #1001 for FlexBeat Lite Earbuds has shipped and will arrive in 2–3 business days.",
    "1002": "Order #1002 for StudyPro 14 Laptop is currently being processed and should ship within 1–2 business days.",
    "1003": "Order #1003 for AeroPods Fit was delivered yesterday."
}

In [ ]:
def detect_intent(user_input):
    text = user_input.lower()

    if "order" in text or "tracking" in text or "track" in text or re.search(r"#?\d{4,}", text):
        return "order_status"
    elif "refund" in text or "return" in text or "money back" in text:
        return "refund_policy"
    else:
        return "product_recommendation"


def extract_order_number(user_input):
    match = re.search(r"\b\d{4,}\b", user_input)
    if match:
        return match.group(0)
    return None


def get_order_status(user_input):
    order_number = extract_order_number(user_input)
    if not order_number:
        return "Please provide your order number so I can check the status."
    return order_database.get(order_number, f"Sorry, I couldn't find order #{order_number} in the system.")


def get_refund_policy():
    return refund_policy_text


def get_product_matches(user_input):
    text = user_input.lower()
    matches = []

    budget = None
    budget_match = re.search(r"under\s*\$?(\d+)|below\s*\$?(\d+)|budget\s*\$?(\d+)", text)
    if budget_match:
        for group in budget_match.groups():
            if group:
                budget = int(group)
                break

    for product in products:
        category_match = product["category"] in text
        use_match = product["best_for"] in text
        feature_match = any(feature in text for feature in product["features"])

        if category_match or use_match or feature_match:
            if budget is None or product["price"] <= budget:
                matches.append(product)

    if not matches:
        for product in products:
            if budget is None or product["price"] <= budget:
                matches.append(product)

    return matches[:3]

In [ ]:
chat_history = []


def format_history(history):
    if not history:
        return "No previous conversation."
    formatted = []
    for turn in history[-6:]:
        formatted.append(f"{turn['role'].upper()}: {turn['content']}")
    return "\n".join(formatted)


def chatbot(user_input):
    global chat_history

    intent = detect_intent(user_input)

    if intent == "order_status":
        store_info = get_order_status(user_input)
    elif intent == "refund_policy":
        store_info = get_refund_policy()
    else:
        matches = get_product_matches(user_input)
        if matches:
            lines = []
            for p in matches:
                lines.append(
                    f"{p['name']} - ${p['price']} - category: {p['category']} - "
                    f"best for: {p['best_for']} - features: {', '.join(p['features'])}"
                )
            store_info = "Relevant product options:\n" + "\n".join(lines)
        else:
            store_info = "No matching products found."

    prompt = f"""
{system_prompt}

You must follow these rules:
- Only use the products, order details, and refund policy explicitly provided in the store information.
- Do not invent product names, prices, policies, or order updates.
- If recommending products, only mention products listed in the relevant store information.
- If no matching products are found, say so clearly and ask a follow-up question.

Conversation history:
{format_history(chat_history)}

Relevant store information:
{store_info}

Current user message:
{user_input}

Write a helpful assistant response.
"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    assistant_reply = response.text

    chat_history.append({"role": "user", "content": user_input})
    chat_history.append({"role": "assistant", "content": assistant_reply})

    return assistant_reply

# Insert Test Prompts such as: "Order Status", "Refund Policy", " Product recommendation"


In [ ]:
print("TechNest Chatbot is running. Type 'exit' to stop.\n")

while True:
    user_input = input("You: ")

    if user_input.lower() == "exit":
        print("Bot: Thanks for visiting TechNest!")
        break

    reply = chatbot(user_input)
    print(f"Bot: {reply}\n")

TechNest Chatbot is running. Type 'exit' to stop.

You: what is the refund policy on headphones
Bot: Hello! Thanks for reaching out. Here's our refund policy regarding headphones and other products:

You can return items within 30 days of delivery for a full refund, provided the item is unused and in its original packaging. If the headphones have been opened, they may still be eligible for store credit, depending on their condition. Please note that final sale items cannot be returned.

You: exit
Bot: Thanks for visiting TechNest!
